# Fable 5 Qwen Sampling + State Capture (Kaggle T4)

Build training dataset: Qwen proposals + state capture on each unique state from Fable 5.
- Input: Fable 5 data (12,490 rows, 255 shards)
- Output: rows with (state_hash, frontier_action, qwen_proposals, hidden_states at depths 18/24/30)
- Device: Kaggle T4 GPU (state capture), CPU (Qwen)
- Create-only: results/raw/fable_sampling_*.json, state captures as .pt files

In [ ]:
# Install dependencies
!pip install --quiet pyarrow torch transformers safetensors

In [ ]:
import os
os.environ.setdefault("ATEN_CPU_CAPABILITY", "avx2")
import json
import glob
import hashlib
import torch
import pyarrow.parquet as pq
from datetime import datetime, timezone
from pathlib import Path

# Paths
FABLE_DIR = Path("/kaggle/input/fable5")  # mount point for uploaded dataset
OUT_DIR = Path("/kaggle/working/output")
OUT_DIR.mkdir(exist_ok=True)

DEPTHS = [18, 24, 30]
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")

print(f"Fable dir: {FABLE_DIR}")
print(f"Output dir: {OUT_DIR}")
print(f"Torch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load Fable parquet shards
parquet_files = sorted(glob.glob(str(FABLE_DIR / "data" / "train-*.parquet")))
print(f"Found {len(parquet_files)} parquet files")

# Skip 0-byte files
parquet_files = [f for f in parquet_files if os.path.getsize(f) > 0]
print(f"After filtering 0-byte: {len(parquet_files)} files")

# Collect all rows
rows_all = []
for pf in parquet_files:
    try:
        table = pq.read_table(pf, columns=["messages", "target_message_index", "assistant_step", "task", "source_trajectory_id"])
        rows_all.extend(table.to_pylist())
    except Exception as e:
        print(f"Error reading {pf}: {e}")

print(f"Loaded {len(rows_all)} rows from Fable")
if rows_all:
    print(f"Sample row keys: {list(rows_all[0].keys())}")

In [ ]:
# Build state + frontier action pairs
# State = messages up to target_message_index - 1
# Frontier = assistant_step at target_message_index

def render_state(messages, target_idx):
    """Render messages up to (but not including) target_message_index."""
    if not messages or target_idx <= 0:
        return ""
    prefix = messages[:target_idx]
    lines = []
    for m in prefix:
        role = m.get("role", "")
        content = m.get("content", "") or m.get("text", "")
        if role and content:
            lines.append(f"{role}: {content}")
    return "\n".join(lines)

decisions = []
state_hashes = {}  # state_hash -> list of row indices with that state

for i, row in enumerate(rows_all):
    messages = row.get("messages") or []
    target_idx = row.get("target_message_index", 0)
    frontier_action = row.get("assistant_step", "")
    task = row.get("task", "")
    traj_id = row.get("source_trajectory_id", f"fable_{i}")
    
    if not frontier_action:
        continue
    
    state_text = render_state(messages, target_idx)
    state_hash = hashlib.sha256(state_text.encode()).hexdigest()[:16]
    
    dec = {
        "decision_id": f"fable_{i}",
        "state_hash": state_hash,
        "state_text": state_text[:2500],  # truncate for size
        "frontier_action": frontier_action,
        "trajectory_id": traj_id,
        "task": task[:500],
        "row_index": i,
    }
    decisions.append(dec)
    
    if state_hash not in state_hashes:
        state_hashes[state_hash] = []
    state_hashes[state_hash].append(i)

print(f"Built {len(decisions)} decision rows")
print(f"Unique states: {len(state_hashes)}")
print(f"Sample decision: {json.dumps(decisions[0], indent=1)[:400]}")

In [ ]:
# Qwen sampling: generate proposals for each unique state
# This runs on Kaggle CPU (Qwen is CPU-friendly with bfloat16)

from transformers import AutoModelForCausalLM, AutoTokenizer

qwen_path = "/kaggle/input/qwen-models/models--Qwen--Qwen3-4B-Thinking-2507/snapshots/768f209d9ea81521153ed38c47d515654e938aea"
if not Path(qwen_path).exists():
    print(f"ERROR: Qwen model not found at {qwen_path}")
    print("Mount the Qwen model dataset to /kaggle/input/qwen-models")
else:
    print(f"Loading Qwen from {qwen_path}")
    tok = AutoTokenizer.from_pretrained(qwen_path)
    model = AutoModelForCausalLM.from_pretrained(
        qwen_path,
        dtype=torch.bfloat16,
        device_map="cpu",
    )
    model.eval()
    print(f"Qwen loaded: {model.config.model_type} {model.config.hidden_size}d")

In [ ]:
# Generate Qwen proposals for each unique state
# Return top 1 greedy proposal (no sampling for reproducibility)

qwen_proposals = {}  # state_hash -> [proposals]

SYSTEM_PROMPT = """You are a coding assistant. Given the conversation history and the task, what is the next best action? 
Reply with a single JSON object: {\"action\": \"<action name>\", \"arguments\": {\"key\": \"value\"}} and nothing else."""

for state_hash, row_indices in list(state_hashes.items())[:100]:  # Start with 100 unique states
    # Get the first row with this state
    row = decisions[row_indices[0]]
    state_text = row["state_text"]
    
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": state_text},
    ]
    
    ids = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    )
    
    with torch.no_grad():
        gen = model.generate(
            ids, max_new_tokens=200, do_sample=False, temperature=1.0
        )
    
    proposal_text = tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)
    
    qwen_proposals[state_hash] = [proposal_text.strip()]
    
    if len(qwen_proposals) % 10 == 0:
        print(f"Sampled {len(qwen_proposals)} unique states...")

print(f"Total unique states sampled: {len(qwen_proposals)}")
if qwen_proposals:
    first_hash = list(qwen_proposals.keys())[0]
    print(f"Sample proposal for {first_hash}: {qwen_proposals[first_hash][0][:200]}")

In [ ]:
# Capture Qwen hidden states for sampled states
# Use the resident capturer to get states at depths 18, 24, 30

from torch import nn

# Simple hook-based capturer (minimal version for this notebook)
captured_states = {}  # state_hash -> {depth: (query, candidates)}

def capture_batch_states(state_texts, state_hashes_list):
    """Capture hidden states for a batch of states."""
    for state_text, state_hash in zip(state_texts, state_hashes_list):
        if state_hash not in qwen_proposals:
            continue
        
        msgs = [{"role": "user", "content": state_text}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
        
        states = {}  # Will be filled by hooks
        
        def make_hook(d):
            def hook(module, input, output):
                if d in DEPTHS:
                    x = output[0] if isinstance(output, tuple) else output
                    states[d] = x[0].detach().cpu()  # [seq_len, hidden]
            return hook
        
        # Register hooks on decoder layers
        handles = []
        for d in DEPTHS:
            if d < len(model.model.layers):
                h = model.model.layers[d].register_forward_hook(make_hook(d))
                handles.append(h)
        
        with torch.no_grad():
            _ = model(ids)
        
        for h in handles:
            h.remove()
        
        captured_states[state_hash] = states

# Capture for first 100 sampled states
states_to_capture = list(qwen_proposals.keys())[:100]
capture_batch_states(
    [decisions[[h for h in state_hashes.keys() if state_hashes[h][0] < len(decisions)][:len(decisions)]][i]["state_text"] if i < len(decisions) else "" for i in range(len(states_to_capture))],
    states_to_capture
)

print(f"Captured states for {len(captured_states)} unique states")

In [ ]:
# Build output rows: decision + frontier action + Qwen proposals + states + NONE

output_rows = []

for dec in decisions:
    state_hash = dec["state_hash"]
    frontier_action = dec["frontier_action"]
    
    # Candidates
    candidates = []
    
    # Frontier (teacher preference = 1)
    candidates.append({
        "id": f"frontier",
        "text": frontier_action,
        "executed": True,
        "teacher_pref": 1,
        "outcome": None,
    })
    
    # Qwen proposals (unlabelled)
    if state_hash in qwen_proposals:
        for j, prop in enumerate(qwen_proposals[state_hash]):
            candidates.append({
                "id": f"qwen_{j}",
                "text": prop[:500],
                "executed": False,
                "teacher_pref": None,
                "outcome": None,
            })
    
    # NONE / low confidence / abstain
    candidates.append({
        "id": "NONE",
        "text": "not sure / low confidence, escalate",
        "executed": False,
        "teacher_pref": None,
        "outcome": None,
    })
    
    # States (if captured)
    states_dict = {}
    if state_hash in captured_states:
        states_dict = {
            d: captured_states[state_hash][d].numpy().tolist() if d in captured_states[state_hash] else None
            for d in DEPTHS
        }
    
    output_rows.append({
        "decision_id": dec["decision_id"],
        "state_hash": state_hash,
        "trajectory_id": dec["trajectory_id"],
        "source": "fable5",
        "candidates": candidates,
        "captured_states": states_dict,
    })

print(f"Built {len(output_rows)} output rows")
print(f"Sample output row: {json.dumps(output_rows[0], indent=1)[:500]}...")

In [ ]:
# Save output
output_file = OUT_DIR / f"fable_sampling_{stamp}.json"

with open(output_file, "w") as f:
    json.dump({
        "label": "exploratory (no gate)",
        "date": stamp,
        "fable_rows_loaded": len(rows_all),
        "unique_states": len(state_hashes),
        "qwen_proposals_sampled": len(qwen_proposals),
        "captured_states": len(captured_states),
        "output_rows": len(output_rows),
        "depths": DEPTHS,
        "rows": output_rows,
    }, f, indent=1)

print(f"Saved to {output_file}")
print(f"File size: {output_file.stat().st_size / 2**20:.1f} MB")

In [ ]:
# Save states as separate .pt files for efficient loading
for state_hash, states_dict in captured_states.items():
    for d in DEPTHS:
        if d in states_dict:
            state_tensor = torch.tensor(states_dict[d])
            state_file = OUT_DIR / f"state_{state_hash}_d{d}.pt"
            torch.save(state_tensor, state_file)

print(f"Saved {len(captured_states) * len(DEPTHS)} state tensors (.pt files)")
print(f"All done. Download files from /kaggle/working/output/")